# Linear Transformations: Deep Dive (Applied Math · Note 07)

Companion notebook for `07-Linear-Transformations.md`. Every number in the note's worked examples and practice solutions is computed here. Figures are produced by `figures_07.py`.

| Part | Topic | Note section |
|---|---|---|
| A | The class example: a 3×2 matrix as a machine ℝ² → ℝ³ | §2, §4 |
| B | Testing linearity: maps that pass and maps that fail | §3, §5 |
| C | Lines and planes through the origin (and when they collapse) | §6 |
| D | Finding the matrix of a transformation | §7 |
| E | Gallery of 2×2 maps: scale, rotate, reflect, shear, project | §8 |
| F | Composition = matrix product (order matters) | §9 |
| G | Kernel & image, one-to-one & onto | §10 |
| H | Affine maps and homogeneous coordinates | §11 |
| I | Neural-network layers | §12 |
| J | Linear maps beyond ℝⁿ: derivative, transpose, trace, det | §13 |
| K | Practice-problem checks P1–P28 | §18 |

In [1]:
import numpy as np
import sympy as sp
np.set_printoptions(suppress=True, precision=4)

def T_of(A):
    A = np.array(A, dtype=float)
    return lambda x: A @ np.array(x, dtype=float)

## Part A — The class example (board, 7 Oct)

In [2]:
A = np.array([[3, 2], [1, 5], [-1, 1]])
u = np.array([1, 2]); v = np.array([3, 4])
print("A u          =", A @ u)
print("A v          =", A @ v)
print("u + v        =", u + v, "-> A(u+v) =", A @ (u + v), "| Au + Av =", A @ u + A @ v)
print("2 Au         =", 2 * (A @ u))
print("3 Av         =", 3 * (A @ v))
print("2Au + 3Av    =", 2 * (A @ u) + 3 * (A @ v))
print("2u + 3v      =", 2 * u + 3 * v, "-> A(2u+3v) =", A @ (2 * u + 3 * v))

x1, x2, x3 = sp.symbols("x1 x2 x3")
print("\nsymbolic A x =", list(sp.Matrix(A) * sp.Matrix([x1, x2])))
B = sp.Matrix([[1, 1, 1], [1, 0, 2]])
print("2x3 example  =", list(B * sp.Matrix([x1, x2, x3])))

A u          = [ 7 11  1]
A v          = [17 23  1]
u + v        = [4 6] -> A(u+v) = [24 34  2] | Au + Av = [24 34  2]
2 Au         = [14 22  2]
3 Av         = [51 69  3]
2Au + 3Av    = [65 91  5]
2u + 3v      = [11 16] -> A(2u+3v) = [65 91  5]

symbolic A x = [3*x1 + 2*x2, x1 + 5*x2, -x1 + x2]
2x3 example  = [x1 + x2 + x3, x1 + 2*x3]


## Part B — Testing linearity

In [3]:
rng = np.random.default_rng(0)

def is_linear(f, n, trials=200):
    # numerical check of T(a u + b v) = a T(u) + b T(v) on random inputs
    for _ in range(trials):
        uu, vv = rng.normal(size=n), rng.normal(size=n)
        a, b = rng.normal(size=2)
        if not np.allclose(f(a * uu + b * vv), a * f(uu) + b * f(vv)):
            return False
    return True

maps = {
    "T1(x) = (2x1 - x2, x1 + 3x2)": (lambda x: np.array([2 * x[0] - x[1], x[0] + 3 * x[1]]), 2),
    "T2(x) = (x1 + 1, x2)":         (lambda x: np.array([x[0] + 1, x[1]]), 2),
    "T3(x) = (x1**2, x2)":          (lambda x: np.array([x[0] ** 2, x[1]]), 2),
    "T4(x) = (x1 * x2, 0)":         (lambda x: np.array([x[0] * x[1], 0.0]), 2),
    "T5(x) = (|x1|, x2)":           (lambda x: np.array([abs(x[0]), x[1]]), 2),
    "T6(x) = (x2, x1, x1 - x2)":    (lambda x: np.array([x[1], x[0], x[0] - x[1]]), 2),
}
for name, (f, n) in maps.items():
    print(f"{name:32s} T(0) = {f(np.zeros(n))}  linear: {is_linear(f, n)}")

T1(x) = (2x1 - x2, x1 + 3x2)     T(0) = [0. 0.]  linear: True
T2(x) = (x1 + 1, x2)             T(0) = [1. 0.]  linear: False
T3(x) = (x1**2, x2)              T(0) = [0. 0.]  linear: False
T4(x) = (x1 * x2, 0)             T(0) = [0. 0.]  linear: False
T5(x) = (|x1|, x2)               T(0) = [0. 0.]  linear: False
T6(x) = (x2, x1, x1 - x2)        T(0) = [0. 0. 0.]  linear: True


In [4]:
# y = m x vs y = m x + c with m = 2, c = 3
f_lin = lambda x: 2 * x
f_aff = lambda x: 2 * x + 3
print("y = 2x    : f(1) + f(2) =", f_lin(1) + f_lin(2), "| f(3) =", f_lin(3), "| f(0) =", f_lin(0))
print("y = 2x + 3: f(1) + f(2) =", f_aff(1) + f_aff(2), "| f(3) =", f_aff(3), "| f(0) =", f_aff(0))
print("y = 2x + 3: 2 f(1) =", 2 * f_aff(1), "| f(2) =", f_aff(2))
# homogeneity alone implies T(0) = 0 (c = 0)
print("T1(0 * u) =", maps["T1(x) = (2x1 - x2, x1 + 3x2)"][0](0 * np.array([5., -7.])))

y = 2x    : f(1) + f(2) = 6 | f(3) = 6 | f(0) = 0
y = 2x + 3: f(1) + f(2) = 12 | f(3) = 9 | f(0) = 3
y = 2x + 3: 2 f(1) = 10 | f(2) = 7
T1(0 * u) = [0. 0.]


## Part C — Lines and planes through the origin

In [5]:
P1 = np.array([[1, 0], [0, 0]])          # board: T(x1, x2) = (x1, 0)
for y in [1, 2, 3]:
    print(f"(2, {y}) -> {P1 @ np.array([2, y])}")
d = np.array([1, 2])                     # the line x2 = 2 x1
print("line t(1,2) -> t *", P1 @ d, "(a line)")
print("x2-axis t(0,1) -> t *", P1 @ np.array([0, 1]), "(collapses to the origin)")

S = np.array([[1, 1], [0, 1]])
print("shear maps direction (1,2) to", S @ d, "-> line x2 = (2/3) x1")

P3 = np.diag([1, 1, 0]); P32 = np.array([[1, 0, 0], [0, 1, 0]])
p = np.array([4, -1, 7])
print("(x,y,z)->(x,y,0):", P3 @ p, "| (x,y,z)->(x,y):", P32 @ p)
# a plane through the origin (spanned by a, b) maps to span{Pa, Pb}
a = np.array([1, 0, 1]); b = np.array([0, 1, 1])
print("plane span{(1,0,1),(0,1,1)} -> span of", P3 @ a, P3 @ b, "rank", np.linalg.matrix_rank(np.c_[P3 @ a, P3 @ b]))
c = np.array([0, 0, 1]); e = np.array([1, 0, 0])
print("plane span{(0,0,1),(1,0,0)} -> span of", P3 @ c, P3 @ e, "rank", np.linalg.matrix_rank(np.c_[P3 @ c, P3 @ e]), "(a line)")

(2, 1) -> [2 0]
(2, 2) -> [2 0]
(2, 3) -> [2 0]
line t(1,2) -> t * [1 0] (a line)
x2-axis t(0,1) -> t * [0 0] (collapses to the origin)
shear maps direction (1,2) to [3 2] -> line x2 = (2/3) x1
(x,y,z)->(x,y,0): [ 4 -1  0] | (x,y,z)->(x,y): [ 4 -1]
plane span{(1,0,1),(0,1,1)} -> span of [1 0 0] [0 1 0] rank 2
plane span{(0,0,1),(1,0,0)} -> span of [0 0 0] [1 0 0] rank 1 (a line)


## Part D — Finding the matrix of a transformation

In [6]:
def matrix_of(f, n):
    # column j = T(e_j)
    return np.column_stack([f(np.eye(n)[j]) for j in range(n)])

T_board1 = lambda x: np.array([x[0], 0.0])
T_board2 = lambda x: np.array([0.0, x[1]])
print("T(x1,x2) = (x1, 0):\n", matrix_of(T_board1, 2))
print("T(x1,x2) = (0, x2):\n", matrix_of(T_board2, 2))

T_ex = lambda x: np.array([x[0] + 2 * x[1], 3 * x[0], x[1] - x[0]])
M = matrix_of(T_ex, 2)
print("T(x1,x2) = (x1+2x2, 3x1, x2-x1):\n", M)
print("check at (2,5):", T_ex(np.array([2., 5.])), M @ np.array([2, 5]))

T(x1,x2) = (x1, 0):
 [[1. 0.]
 [0. 0.]]
T(x1,x2) = (0, x2):
 [[0. 0.]
 [0. 1.]]
T(x1,x2) = (x1+2x2, 3x1, x2-x1):
 [[ 1.  2.]
 [ 3.  0.]
 [-1.  1.]]
check at (2,5): [12.  6.  3.] [12.  6.  3.]


## Part E — Gallery of 2×2 maps

In [7]:
th = np.deg2rad(30)
gallery = {
    "scale diag(2, 0.5)": np.diag([2, 0.5]),
    "rotate 90°": np.array([[0, -1], [1, 0]]),
    "rotate 30°": np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]),
    "reflect x-axis": np.array([[1, 0], [0, -1]]),
    "reflect y = x": np.array([[0, 1], [1, 0]]),
    "shear [[1,1],[0,1]]": np.array([[1, 1], [0, 1]]),
    "project x1-axis": np.array([[1, 0], [0, 0]]),
}
x = np.array([3, 1])
for name, G in gallery.items():
    print(f"{name:20s} det = {np.linalg.det(G):7.4f}   G(3,1) = {G @ x}   G(1,0) = {G @ np.array([1, 0])}")
print("\nrotation 30° preserves length:", np.linalg.norm(x), np.linalg.norm(gallery['rotate 30°'] @ x))
print("area of image of unit square under [[2,1],[1,3]]:", np.linalg.det(np.array([[2, 1], [1, 3]])))

scale diag(2, 0.5)   det =  1.0000   G(3,1) = [6.  0.5]   G(1,0) = [2. 0.]
rotate 90°           det =  1.0000   G(3,1) = [-1  3]   G(1,0) = [0 1]
rotate 30°           det =  1.0000   G(3,1) = [2.0981 2.366 ]   G(1,0) = [0.866 0.5  ]
reflect x-axis       det = -1.0000   G(3,1) = [ 3 -1]   G(1,0) = [1 0]
reflect y = x        det = -1.0000   G(3,1) = [1 3]   G(1,0) = [0 1]
shear [[1,1],[0,1]]  det =  1.0000   G(3,1) = [4 1]   G(1,0) = [1 0]
project x1-axis      det =  0.0000   G(3,1) = [3 0]   G(1,0) = [1 0]

rotation 30° preserves length: 3.1622776601683795 3.162277660168379
area of image of unit square under [[2,1],[1,3]]: 5.000000000000001


## Part F — Composition = matrix product

In [8]:
R = gallery["rotate 90°"]; F = gallery["reflect x-axis"]
print("rotate then reflect = F @ R =\n", F @ R)
print("reflect then rotate = R @ F =\n", R @ F)
x = np.array([2, 1])
print("x = (2,1): F(R x) =", F @ (R @ x), "| (F R) x =", (F @ R) @ x, "| R(F x) =", R @ (F @ x))
R30 = gallery["rotate 30°"]
print("R30 applied 3 times = rotate 90°:", np.allclose(np.linalg.matrix_power(R30, 3), R))
P = gallery["project x1-axis"]
print("project twice = project once:", np.array_equal(P @ P, P))

rotate then reflect = F @ R =
 [[ 0 -1]
 [-1  0]]
reflect then rotate = R @ F =
 [[0 1]
 [1 0]]
x = (2,1): F(R x) = [-1 -2] | (F R) x = [-1 -2] | R(F x) = [1 2]
R30 applied 3 times = rotate 90°: True
project twice = project once: True


## Part G — Kernel & image

In [9]:
for name, M in [("A (3x2, class)", sp.Matrix([[3, 2], [1, 5], [-1, 1]])),
                ("B (2x3, class)", sp.Matrix([[1, 1, 1], [1, 0, 2]])),
                ("P (projection)", sp.Matrix([[1, 0], [0, 0]]))]:
    m, n = M.shape
    r = M.rank()
    print(f"{name}: R^{n} -> R^{m}, rank {r}, nullity {n - r}, kernel {[list(k) for k in M.nullspace()]}, "
          f"one-to-one {n - r == 0}, onto {r == m}")

Acl = np.array([[3, 2], [1, 5], [-1, 1]])
normal = np.cross(Acl[:, 0], Acl[:, 1])
print("\nimage of A is the plane with normal", normal)
for b in [(7, 11, 1), (1, 1, 1), (65, 91, 5)]:
    print(f"b = {b}: normal . b = {normal @ np.array(b)} -> in image: {normal @ np.array(b) == 0}")

A (3x2, class): R^2 -> R^3, rank 2, nullity 0, kernel [], one-to-one True, onto False
B (2x3, class): R^3 -> R^2, rank 2, nullity 1, kernel [[-2, 1, 1]], one-to-one False, onto True
P (projection): R^2 -> R^2, rank 1, nullity 1, kernel [[0, 1]], one-to-one False, onto False

image of A is the plane with normal [ 6 -5 13]
b = (7, 11, 1): normal . b = 0 -> in image: True
b = (1, 1, 1): normal . b = 14 -> in image: False
b = (65, 91, 5): normal . b = 0 -> in image: True


## Part H — Affine maps and homogeneous coordinates

In [10]:
H1 = np.array([[2, 3], [0, 1]])           # f(x) = 2x + 3 as a 2x2 matrix acting on (x, 1)
print("f(5) via [[2,3],[0,1]] @ (5,1) =", H1 @ np.array([5, 1]))
Tr = np.array([[1, 0, 2], [0, 1, -1], [0, 0, 1]])   # translate by (2, -1)
print("translate (1,1) by (2,-1):", Tr @ np.array([1, 1, 1]))
Rh = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])   # rotate 90° in homogeneous form
print("rotate then translate (1,1):", (Tr @ Rh) @ np.array([1, 1, 1]),
      "| translate then rotate:", (Rh @ Tr) @ np.array([1, 1, 1]))

f(5) via [[2,3],[0,1]] @ (5,1) = [13  1]
translate (1,1) by (2,-1): [3 0 1]
rotate then translate (1,1): [1 0 1] | translate then rotate: [0 3 1]


## Part I — Neural-network layers

In [11]:
W1 = np.array([[1, 2], [0, 1], [1, -1]]); W2 = np.array([[1, 1, 2]])
x = np.array([1, 2])
print("W2 W1 =", W2 @ W1, "| two linear layers on x:", W2 @ (W1 @ x), "| collapsed:", (W2 @ W1) @ x)
h = W1 @ x
print("W1 x =", h, "| ReLU =", np.maximum(h, 0), "| W2 ReLU(W1 x) =", W2 @ np.maximum(h, 0))
relu = lambda z: np.maximum(z, 0)
print("ReLU linear?  ReLU(-1 * (1)) =", relu(-1.0), " vs -1 * ReLU(1) =", -1 * relu(1.0))

W2 W1 = [[3 1]] | two linear layers on x: [5] | collapsed: [5]
W1 x = [ 5  2 -1] | ReLU = [5 2 0] | W2 ReLU(W1 x) = [7]
ReLU linear?  ReLU(-1 * (1)) = 0.0  vs -1 * ReLU(1) = -1.0


## Part J — Linear maps beyond ℝⁿ

In [12]:
D = sp.Matrix([[0, 1, 0], [0, 0, 2], [0, 0, 0]])   # d/dx on P2 in basis {1, x, x^2}
p = sp.Matrix([3, 2, 5])                           # 3 + 2x + 5x^2
print("D p =", list(D * p), "-> 2 + 10x")
xs = sp.symbols("x")
print("sympy check:", sp.diff(3 + 2 * xs + 5 * xs**2, xs))
M = sp.Matrix([[1, 2], [3, 4]])
print("det(2M) =", (2 * M).det(), "vs 2 det(M) =", 2 * M.det(), "-> det is not linear")
print("trace(2M + N) = 2 trace(M) + trace(N):", (2 * M + sp.eye(2)).trace() == 2 * M.trace() + sp.eye(2).trace())

D p = [2, 10, 0] -> 2 + 10x
sympy check: 10*x + 2
det(2M) = -8 vs 2 det(M) = -4 -> det is not linear
trace(2M + N) = 2 trace(M) + trace(N): True


## Part K — Practice-problem checks

In [13]:
print("P1 :", matrix_of(lambda x: np.array([x[0] + x[1], 3 * x[0]]), 2).tolist())
print("P5 :", A @ np.array([5, -1]))
Tu, Tv = np.array([1, 2]), np.array([0, -1])
print("P6 :", 3 * Tu - 2 * Tv)
M7 = np.column_stack([[2, 1], [-1, 3]])
print("P7 :", M7.tolist(), M7 @ np.array([4, 5]))
M8 = np.column_stack([[3, 1], [1, -1]]) @ np.linalg.inv(np.column_stack([[1, 1], [1, -1]]))
print("P8 :", M8.round(10).tolist())
print("P11:", gallery["rotate 90°"] @ np.array([3, 1]))
print("P12:", (gallery["rotate 30°"] @ np.array([1, 0])).round(4))
Sh = np.array([[1, 2], [0, 1]])
sq = np.array([[0, 0], [1, 0], [1, 1], [0, 1]]).T
print("P13:", (Sh @ sq).T.tolist(), "det", np.linalg.det(Sh))
print("P14: F R =", (F @ R).tolist(), "| R F =", (R @ F).tolist())
print("P16:", np.array([[1, 1], [0, 1]]) @ np.array([1, 2]))
print("P19:", normal, normal @ np.array([1, 1, 1]), normal @ np.array([7, 11, 1]))
print("P21:", Tr @ np.array([1, 1, 1]))
print("P22:", (W2 @ W1).tolist(), (W2 @ W1) @ x, W2 @ relu(W1 @ x))
print("P27:", A @ (2 * u + 3 * v))
print("P28:", np.linalg.det(np.array([[2, 1], [1, 3]])).round(10))

P1 : [[1.0, 1.0], [3.0, 0.0]]
P5 : [13  0 -6]
P6 : [3 8]
P7 : [[2, -1], [1, 3]] [ 3 19]
P8 : [[2.0, 1.0], [0.0, 1.0]]
P11: [-1  3]
P12: [0.866 0.5  ]
P13: [[0, 0], [1, 0], [3, 1], [2, 1]] det 1.0
P14: F R = [[0, -1], [-1, 0]] | R F = [[0, 1], [1, 0]]
P16: [3 2]
P19: [ 6 -5 13] 14 0
P21: [3 0 1]
P22: [[3, 1]] [5] [7]
P27: [65 91  5]
P28: 5.0
